# 01 高光谱数据读取、划分与预处理

本 Notebook 是实验流程的**第一阶段**，完成以下工作：

1. 配置运行环境与路径，读取 **Pavia University / Indian Pines / Salinas** 三套数据集；
2. 可视化查看数据效果，分析数据集类别构成；
3. 完成**分层随机划分**（训练 / 验证 / 测试 = 24% / 6% / 70%）；
4. 对数据做 **逐波段标准化 + 降维**（原始 / PCA / LDA / 波段选择四种方式），并演示邻域 patch 提取；
5. 输出模型就绪数据与配置文件（manifest），供第二阶段 HybridSN 训练、第三阶段传统方法对比直接读取。

> 约定：所有统计量（标准化均值/方差、PCA/LDA 变换矩阵、波段选择得分）都**只在训练集上拟合**，再变换整幅图像，避免测试集信息泄漏。

In [ ]:
# ==================== 0. 环境配置与路径验证 ====================
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# 中文与负号正常显示
matplotlib.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

# ---- 路径配置（按需修改）----
# 项目根目录（实验交付/）：向上查找含 configs/ 与 README.md 的目录，
# 保证无论从何处启动 Jupyter（notebooks/ 或 实验交付/）都能正确定位
PROJECT_ROOT = Path.cwd().resolve()
for _p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (_p / "configs").is_dir() and (_p / "README.md").is_file():
        PROJECT_ROOT = _p
        break

# 原始 .mat 数据目录（三套数据集共同存放；体积大未复制，直接引用）
RAW_DATA_DIR = Path(r"g:/高光谱智能图像解译/实验交付/data/raw")
if not RAW_DATA_DIR.is_dir():
    RAW_DATA_DIR = PROJECT_ROOT.parent / "实验交付" / "data" / "raw"

# 划分与预处理产物输出目录
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "stage1"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"   # 汇总可视化图输出目录（供报告引用）

# ---- 随机种子----
SEED = 1442

# ---- 验证数据文件是否存在 ----
required = ["PaviaU.mat", "PaviaU_gt.mat", "Indian_pines_corrected.mat",
            "Indian_pines_gt.mat", "Salinas_corrected.mat", "Salinas_gt.mat"]
missing = [f for f in required if not (RAW_DATA_DIR / f).is_file()]
assert not missing, f"缺少数据文件: {missing}"
print("数据文件齐全 [OK]")
print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW_DATA_DIR =", RAW_DATA_DIR)

import torch, sklearn
print("Python", sys.version.split()[0], "| torch", torch.__version__,
      "| sklearn", sklearn.__version__)

## 1. 数据集加载与简单取样查看

先建立三套数据集的元信息注册表（文件名、MAT 变量名、类别名），再统一读取并打印形状信息。

In [ ]:
# ==================== 1. 数据集加载与简单取样查看 ====================
# 三套数据集的元信息注册表：文件名称、MAT 变量名、类别名
DATASETS = {
    "pavia_university": {
        "data_file": "PaviaU.mat", "label_file": "PaviaU_gt.mat",
        "data_key": "paviaU", "label_key": "paviaU_gt",
        "class_names": ["Asphalt", "Meadows", "Gravel", "Trees",
                        "Painted metal sheets", "Bare Soil", "Bitumen",
                        "Self-Blocking Bricks", "Shadows"],
    },
    "indian_pines": {
        "data_file": "Indian_pines_corrected.mat", "label_file": "Indian_pines_gt.mat",
        "data_key": "indian_pines_corrected", "label_key": "indian_pines_gt",
        "class_names": ["Alfalfa", "Corn-notill", "Corn-mintill", "Corn",
                        "Grass-pasture", "Grass-trees", "Grass-pasture-mowed",
                        "Hay-windrowed", "Oats", "Soybean-notill", "Soybean-mintill",
                        "Soybean-clean", "Wheat", "Woods",
                        "Buildings-Grass-Trees-Drives", "Stone-Steel-Towers"],
    },
    "salinas": {
        "data_file": "Salinas_corrected.mat", "label_file": "Salinas_gt.mat",
        "data_key": "salinas_corrected", "label_key": "salinas_gt",
        "class_names": ["Broccoli_green_weeds_1", "Broccoli_green_weeds_2",
                        "Fallow", "Fallow_rough_plow", "Fallow_smooth", "Stubble",
                        "Celery", "Grapes_untrained", "Soil_vinyard_develop",
                        "Corn_senesced_green_weeds", "Lettuce_romaine_4wk",
                        "Lettuce_romaine_5wk", "Lettuce_romaine_6wk",
                        "Lettuce_romaine_7wk", "Vinyard_untrained",
                        "Vinyard_vertical_trellis"],
    },
}


def load_dataset(name):
    # 读取影像立方体（H×W×B）与地物真值图（H×W）
    meta = DATASETS[name]
    cube = np.asarray(loadmat(RAW_DATA_DIR / meta["data_file"])[meta["data_key"]],
                      dtype=np.float32)
    label = np.asarray(loadmat(RAW_DATA_DIR / meta["label_file"])[meta["label_key"]])
    return cube, label, meta["class_names"]


# 逐套打印形状与类别信息
for name in DATASETS:
    cube, label, class_names = load_dataset(name)
    classes = np.unique(label)
    print(f"{name:18s} 影像 {cube.shape}  真值 {label.shape}  "
          f"类别 {classes.min()}~{classes.max()}（{len(class_names)}类）  "
          f"标记像元 {np.count_nonzero(label)}")

下面以假彩色合成图与地物真值图直观查看三套数据集的空间结构。

In [ ]:
# 对每套数据集绘制假彩色合成图（RGB 取三个代表性波段）与地物真值图
def show_overview(name, rgb_bands):
    cube, label, class_names = load_dataset(name)
    # 对三个波段做 2%~98% 百分位拉伸，改善显示对比度
    def stretch(band):
        lo, hi = np.percentile(band, 2), np.percentile(band, 98)
        return np.clip((band - lo) / (hi - lo + 1e-6), 0, 1)
    rgb = np.stack([stretch(cube[:, :, b]) for b in rgb_bands], axis=-1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    axes[0].imshow(rgb)
    axes[0].set_title(f"{name}  假彩色合成（波段 {rgb_bands}）")
    axes[0].axis("off")
    im = axes[1].imshow(label, cmap="nipy_spectral")
    axes[1].set_title(f"{name}  地物真值（{len(class_names)} 类）")
    axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(FIG_DIR / f"{name}_overview.png", dpi=150, bbox_inches="tight")
    plt.show()

# 三套数据集：波段索引按“前/中/后”选取，保证 RGB 视觉区分度
show_overview("pavia_university", [60, 30, 10])
show_overview("indian_pines", [60, 30, 10])
show_overview("salinas", [60, 30, 10])

## 2. 数据集构成分析

统计每套数据集各类别的样本数量，观察类别不平衡程度。

In [ ]:
# ==================== 2. 数据集构成分析 ====================
# 统计每套数据集各类别的样本数量
for name in DATASETS:
    cube, label, class_names = load_dataset(name)
    ids = np.arange(1, len(class_names) + 1)
    counts = np.array([np.count_nonzero(label == c) for c in ids])
    print(f"\n===== {name} =====")
    for c, n in zip(class_names, counts):
        print(f"  类别 {ids[class_names.index(c)]:2d}  {c:32s} {n:6d}")

# 对关注的数据集（默认 Pavia University）绘制类别分布柱状图
FOCUS = "pavia_university"          # 可改为 indian_pines / salinas
cube, label, class_names = load_dataset(FOCUS)
ids = np.arange(1, len(class_names) + 1)
counts = np.array([np.count_nonzero(label == c) for c in ids])
fig, ax = plt.subplots(figsize=(10, 4.4))
bars = ax.bar(ids, counts, color="#2563EB")
ax.set_xticks(ids)
ax.set_xticklabels([f"{i}\n{n}" for i, n in zip(ids, class_names)], fontsize=7)
ax.set_ylabel("样本数量")
ax.set_title(f"{FOCUS}  各类别样本数量分布（共 {counts.sum()} 个标记像元）")
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(n),
            ha="center", va="bottom", fontsize=7)
plt.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / f"{FOCUS}_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 数据集划分

采用**分层随机划分**，训练 / 验证 / 测试 = **24% / 6% / 70%**：

- 先按 30% / 70% 划分出「训练池 / 测试集」；
- 再从 30% 训练池中按 80% / 20% 划分出「训练集 / 验证集」；
- 最终得到 24% / 6% / 70%。

划分结果保存到 `data/splits/`，第二阶段、第三阶段直接读取。

In [ ]:
# ==================== 3. 数据集划分 ====================
def create_fixed_split(label, seed=SEED):
    """返回坐标、标签与三份索引（分层随机）。"""
    coords = np.argwhere(label != 0).astype(np.int32)
    labels = label[coords[:, 0], coords[:, 1]].astype(np.int16)
    all_idx = np.arange(labels.size)
    train_pool, test_idx = train_test_split(
        all_idx, train_size=0.30, test_size=0.70,
        random_state=seed, shuffle=True, stratify=labels)
    train_idx, val_idx = train_test_split(
        train_pool, train_size=0.80, test_size=0.20,
        random_state=seed, shuffle=True, stratify=labels[train_pool])
    return coords, labels, train_idx, val_idx, test_idx

# 对三套数据集生成并保存划分
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    _, label, class_names = load_dataset(name)
    coords, labels, train_idx, val_idx, test_idx = create_fixed_split(label, SEED)
    np.savez_compressed(
        SPLIT_DIR / f"{name}__fair24_6_70__seed{SEED}.npz",
        coordinates=coords, labels=labels,
        train_indices=train_idx, validation_indices=val_idx, test_indices=test_idx)
    info = {"dataset": name, "protocol": "fair24_6_70", "seed": SEED,
            "class_names": class_names,
            "train": int(train_idx.size), "validation": int(val_idx.size),
            "test": int(test_idx.size)}
    (SPLIT_DIR / f"{name}__fair24_6_70__seed{SEED}.json").write_text(
        json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{name}: train={train_idx.size}  validation={val_idx.size}  test={test_idx.size}")

# 可视化关注数据集的划分空间分布（训练=红、验证=绿、测试=蓝、背景=白）
cube, label, class_names = load_dataset(FOCUS)
coords, labels, train_idx, val_idx, test_idx = create_fixed_split(label, SEED)
split_map = np.zeros(label.shape, dtype=np.uint8)
split_map[coords[train_idx, 0], coords[train_idx, 1]] = 1
split_map[coords[val_idx, 0],   coords[val_idx, 1]]   = 2
split_map[coords[test_idx, 0],  coords[test_idx, 1]]  = 3
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["white", "#E11D48", "#10B981", "#2563EB"])
plt.figure(figsize=(6.2, 5))
plt.imshow(split_map, cmap=cmap)
plt.title(f"{FOCUS}  划分空间分布（红=训练 绿=验证 蓝=测试）")
plt.axis("off")
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / f"{FOCUS}_split_map.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. 降维处理（原始 / PCA / LDA / 波段选择）

处理流程为「**逐波段标准化 → 降维**」：

- `raw`：原始波段（仅标准化，不降维）；
- `pca`：主成分分析（无监督线性降维）；
- `lda`：线性判别分析（有监督，利用类别信息，维度 ≤ 类别数 − 1）；
- `band_selection`：波段选择（`uniform` 等间隔均匀选取 / `fisher` 按判别得分排序选取）。

所有变换的统计量都只在**训练集**上拟合。

In [ ]:
# ==================== 4. 降维处理（原始 / PCA / LDA / 波段选择） ====================
def fit_standardizer(train_spectra):
    """在训练集上拟合逐波段标准化参数（均值 / 标准差）。"""
    mean = train_spectra.mean(axis=0, dtype=np.float64)
    scale = train_spectra.std(axis=0, dtype=np.float64)
    scale[scale == 0] = 1.0
    return mean, scale

def apply_reducer(std_cube, train_std, train_labels, reducer, n_components=None, method="fisher"):
    """按指定方式在训练集上拟合并变换整幅影像，返回 (降维立方体, 附加信息)。"""
    info = {}
    if reducer == "none":
        return std_cube, info
    if reducer == "pca":
        pca = PCA(n_components=n_components, whiten=False)
        pca.fit(train_std)
        info["explained_variance_ratio"] = pca.explained_variance_ratio_
        red = pca.transform(std_cube.reshape(-1, std_cube.shape[2]))
        return red.reshape(std_cube.shape[0], std_cube.shape[1], n_components), info
    if reducer == "lda":
        lda = LinearDiscriminantAnalysis(n_components=n_components)
        lda.fit(train_std, train_labels)
        info["explained_variance_ratio"] = lda.explained_variance_ratio_
        red = lda.transform(std_cube.reshape(-1, std_cube.shape[2]))
        return red.reshape(std_cube.shape[0], std_cube.shape[1], n_components), info
    if reducer == "band_selection":
        if method == "uniform":
            selected = np.rint(np.linspace(0, std_cube.shape[2] - 1, n_components)).astype(int)
            scores = np.full(std_cube.shape[2], np.nan)
        else:  # fisher：类间散度 / 类内散度
            classes = np.unique(train_labels)
            global_mean = train_std.mean(axis=0)
            between = np.zeros(std_cube.shape[2])
            within = np.zeros(std_cube.shape[2])
            for c in classes:
                cv = train_std[train_labels == c]
                cm = cv.mean(axis=0)
                between += cv.shape[0] * (cm - global_mean) ** 2
                within += ((cv - cm) ** 2).sum(axis=0)
            scores = between / np.maximum(within, np.finfo(np.float64).eps)
            selected = np.sort(np.argsort(-scores, kind="mergesort")[:n_components])
        info["selected_bands"] = selected
        info["scores"] = scores
        return std_cube[:, :, selected], info
    raise ValueError(f"未知降维方式: {reducer}")

对关注数据集跑全部预处理路线，并可视化 PCA 解释方差、PCA 散点与波段选择得分。

In [ ]:
# 对关注数据集跑全部预处理路线，并可视化 PCA 解释方差、LDA 散点、波段选择得分
cube, label, class_names = load_dataset(FOCUS)
coords = np.argwhere(label != 0).astype(np.int32)
labels = label[coords[:, 0], coords[:, 1]].astype(np.int16)
_, _, train_idx, val_idx, test_idx = create_fixed_split(label, SEED)

# 训练像元光谱（用于拟合所有统计量）
train_spectra = cube[coords[train_idx, 0], coords[train_idx, 1], :]
mean, scale = fit_standardizer(train_spectra)
std_cube = (cube - mean) / scale
train_std = (train_spectra - mean) / scale

# 依次计算五种路线（pca15 / lda 维度自动取类别数-1 / uniform15 / fisher15 + raw）
LDA_N = len(class_names) - 1        # LDA 维度上限 = 类别数 - 1
routes = {
    "raw_all_bands": dict(reducer="none"),
    "pca15": dict(reducer="pca", n_components=15),
    f"lda{LDA_N}": dict(reducer="lda", n_components=LDA_N),
    "uniform15": dict(reducer="band_selection", n_components=15, method="uniform"),
    "fisher15": dict(reducer="band_selection", n_components=15, method="fisher"),
}
reduced, route_info = {}, {}
for key, kw in routes.items():
    reduced[key], route_info[key] = apply_reducer(std_cube, train_std, labels[train_idx], **kw)
    print(f"{key:16s} -> {reduced[key].shape}")

# —— 可视化：PCA 解释方差累计曲线 ——
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
evr = route_info["pca15"]["explained_variance_ratio"]
axes[0].plot(np.arange(1, len(evr) + 1), np.cumsum(evr), "o-", color="#2563EB")
axes[0].set_title("PCA 累计解释方差")
axes[0].set_xlabel("主成分个数"); axes[0].set_ylabel("累计解释方差比")

# —— 可视化：PCA 前两主成分散点（按类别着色）——
pca2 = PCA(n_components=2).fit_transform(train_std)
sc = axes[1].scatter(pca2[:, 0], pca2[:, 1], c=labels[train_idx], cmap="nipy_spectral", s=4)
axes[1].set_title("PCA 前两主成分散点（训练集）")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
fig.colorbar(sc, ax=axes[1], fraction=0.046)

# —— 可视化：Fisher 波段选择得分 ——
scores = route_info["fisher15"]["scores"]
axes[2].plot(np.arange(1, scores.size + 1), scores, color="#F59E0B")
selected = route_info["fisher15"]["selected_bands"]
axes[2].scatter(selected + 1, scores[selected], color="#E11D48", zorder=3, label="被选波段")
axes[2].set_title("Fisher 判别得分（波段选择）")
axes[2].set_xlabel("波段索引"); axes[2].set_ylabel("得分")
axes[2].legend()
plt.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / f"{FOCUS}_reduction_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

演示邻域 patch 提取：以每个标记像元为中心取 `patch_size × patch_size` 邻域块，边缘不足时补零，与 HybridSN 论文一致。

In [ ]:
# 邻域 patch 提取：以每个标记像元为中心取 patch_size×patch_size 邻域块
PATCH_SIZE = 25                       # 邻域块边长（可调，奇数）
def extract_patch(reduced_cube, row, col, patch_size=PATCH_SIZE):
    r = patch_size // 2
    pad = np.pad(reduced_cube, ((r, r), (r, r), (0, 0)), mode="constant")
    return pad[row:row + patch_size, col:col + patch_size, :].transpose(2, 0, 1)[None, ...]

# 演示：取训练集第一个像元，展示其 PCA15 patch（前 3 个主成分作 RGB）
demo = reduced["pca15"]
row, col = coords[train_idx[0]]
patch = extract_patch(demo, row, col)
print(f"中心像元坐标=({row},{col})  类别={labels[train_idx[0]]}  patch 形状={patch.shape}")
plt.figure(figsize=(5, 4.6))
plt.imshow(patch[0, :3].transpose(1, 2, 0))
plt.title("PCA15 patch 前 3 主成分（25×25）")
plt.axis("off")
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / f"{FOCUS}_patch_example.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. 输出预处理配置文件

将「降维后的整幅立方体 + 划分 + 元信息」保存为模型就绪 `.npz`，并写出 `stage1_manifest.json`（记录默认路线与各路线路径），供第二阶段、第三阶段直接读取。

In [ ]:
# ==================== 5. 输出预处理配置文件 ====================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_base = OUTPUT_DIR / FOCUS
out_base.mkdir(parents=True, exist_ok=True)

manifest_routes = {}
for key, kw in routes.items():
    npz_path = out_base / f"{key}.npz"
    np.savez_compressed(
        npz_path,
        transformed_cube=reduced[key].astype(np.float32),
        coordinates=coords,
        raw_labels=labels,
        train_indices=train_idx,
        validation_indices=val_idx,
        test_indices=test_idx,
        class_names=np.array(class_names),
        patch_size=np.array(PATCH_SIZE),
        num_classes=np.array(len(class_names)),
    )
    manifest_routes[key] = str(npz_path.relative_to(PROJECT_ROOT))
    print(f"已保存 {npz_path.relative_to(PROJECT_ROOT)}")

SELECTED_ROUTE = "pca15"              # 下游默认使用的路线（可改：raw_all_bands / lda 等）
manifest = {
    "dataset": FOCUS,
    "protocol": "fair24_6_70",
    "seed": SEED,
    "selected_route": SELECTED_ROUTE,
    "selected_artifact": manifest_routes[SELECTED_ROUTE],
    "routes": manifest_routes,
    "class_names": class_names,
    "patch_size": PATCH_SIZE,
}
manifest_path = out_base / "stage1_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("manifest ->", manifest_path.relative_to(PROJECT_ROOT))
print("\n第一阶段完成。第二阶段/第三阶段将读取 stage1_manifest.json 中的模型就绪数据。")